In [ ]:
# -*- coding: utf-8 -*-
"""
Ανάλυση Δεδομένων Τρεξίματος από Garmin Forerunner 245

Σκοπός αυτού του notebook είναι να αναλύσει δεδομένα τρεξίματος που έχουν εξαχθεί από ένα ρολόι
Garmin Forerunner 245 και αποθηκευτεί σε μια βάση δεδομένων SQLite χρησιμοποιώντας το project garmindb.

Θα πραγματοποιήσουμε τις ακόλουθες αναλύσεις:
1. Βασικές Μετρήσεις και Τάσεις (Ρυθμός, Απόσταση, Διάρκεια)
2. Ανάλυση Καρδιακών Παλμών (Μέσος Παλμός, Ζώνες Παλμών)
3. Ανάλυση Ρυθμού και Υψομέτρου
4. Στατιστικά Laps (αν υπάρχουν)
5. Μακροπρόθεσμες Τάσεις και Συγκρίσεις
"""

# ## Εισαγωγή Βιβλιοθηκών

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ρυθμίσεις για καλύτερες οπτικοποιήσεις
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# ## Σύνδεση στη Βάση Δεδομένων SQLite

# **ΠΡΟΣΟΧΗ:** Αντικαταστήστε την παρακάτω διαδρομή με την πραγματική διαδρομή προς το αρχείο της βάσης δεδομένων σας.
DATABASE_PATH = 'path/to/your/garmin_data.db'

def connect_to_db(db_path):
    """Συνδέεται με τη βάση δεδομένων SQLite."""
    try:
        conn = sqlite3.connect(db_path)
        return conn
    except sqlite3.Error as e:
        print(f"Σφάλμα σύνδεσης με τη βάση δεδομένων: {e}")
        return None

conn = connect_to_db(DATABASE_PATH)

if conn:
    cursor = conn.cursor()
    print(f"Επιτυχής σύνδεση με τη βάση δεδομένων: {DATABASE_PATH}")
else:
    exit()

# ## 1. Βασικές Μετρήσεις και Τάσεις

# ### Μέσος Ρυθμός ανά Τρέξιμο

query_avg_pace = """
SELECT
    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,
    AVG(1000.0 / r.speed) / 60.0 AS avg_pace_min_km
FROM activity a
JOIN record r ON a.activity_id = r.activity_id
WHERE a.sport = 'running' AND r.speed > 0
GROUP BY run_date
ORDER BY run_date;
"""
df_pace = pd.read_sql_query(query_avg_pace, conn, parse_dates=['run_date'])

if not df_pace.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(df_pace['run_date'], df_pace['avg_pace_min_km'], marker='o', linestyle='-')
    plt.title('Μέσος Ρυθμός Τρεξίματος με την Πάροδο του Χρόνου')
    plt.xlabel('Ημερομηνία')
    plt.ylabel('Μέσος Ρυθμός (λεπτά/km)')
    plt.grid(True)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Δεν βρέθηκαν δεδομένα τρεξίματος για ανάλυση ρυθμού.")

# ### Μέση Απόσταση ανά Τρέξιμο

query_distance = """
SELECT
    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,
    a.total_distance / 1000.0 AS distance_km
FROM activity a
WHERE a.sport = 'running' AND a.total_distance > 0
ORDER BY run_date;
"""
df_distance = pd.read_sql_query(query_distance, conn, parse_dates=['run_date'])

if not df_distance.empty:
    plt.figure(figsize=(12, 6))
    plt.bar(df_distance['run_date'], df_distance['distance_km'])
    plt.title('Απόσταση Τρεξίματος ανά Ημερομηνία')
    plt.xlabel('Ημερομηνία')
    plt.ylabel('Απόσταση (km)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Δεν βρέθηκαν δεδομένα τρεξίματος για ανάλυση απόστασης.")

# ### Μέση Διάρκεια ανά Τρέξιμο

query_duration = """
SELECT
    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,
    a.total_duration / 60.0 AS duration_minutes
FROM activity a
WHERE a.sport = 'running' AND a.total_duration > 0
ORDER BY run_date;
"""
df_duration = pd.read_sql_query(query_duration, conn, parse_dates=['run_date'])

if not df_duration.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(df_duration['run_date'], df_duration['duration_minutes'], marker='o', linestyle='-')
    plt.title('Διάρκεια Τρεξίματος με την Πάροδο του Χρόνου')
    plt.xlabel('Ημερομηνία')
    plt.ylabel('Διάρκεια (λεπτά)')
    plt.grid(True)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Δεν βρέθηκαν δεδομένα τρεξίματος για ανάλυση διάρκειας.")

# ## 2. Ανάλυση Καρδιακών Παλμών

# ### Μέσος Καρδιακός Παλμός ανά Τρέξιμο

query_avg_hr = """
SELECT
    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,
    AVG(r.heart_rate) AS avg_heart_rate
FROM activity a
JOIN record r ON a.activity_id = r.activity_id
WHERE a.sport = 'running' AND r.heart_rate > 0
GROUP BY run_date
ORDER BY run_date;
"""
df_hr = pd.read_sql_query(query_avg_hr, conn, parse_dates=['run_date'])

if not df_hr.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(df_hr['run_date'], df_hr['avg_heart_rate'], marker='o', linestyle='-')
    plt.title('Μέσος Καρδιακός Παλμός ανά Τρέξιμο')
    plt.xlabel('Ημερομηνία')
    plt.ylabel('Μέσος Καρδιακός Παλμός (bpm)')
    plt.grid(True)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Δεν βρέθηκαν δεδομένα καρδιακών παλμών για τρέξιμο.")

# ### Κατανομή Χρόνου σε Ζώνες Καρδιακών Παλμών

def calculate_hr_zones(heart_rates, max_hr):
    """Υπολογίζει το ποσοστό του χρόνου που πέρασε σε κάθε ζώνη καρδιακών παλμών."""
    zones = {'Zone 1': 0, 'Zone 2': 0, 'Zone 3': 0, 'Zone 4': 0, 'Zone 5': 0}
    total_time = len(heart_rates)
    if total_time > 0:
        for hr in heart_rates:
            if hr < 0.6 * max_hr:
                zones['Zone 1'] += 1
            elif hr < 0.7 * max_hr:
                zones['Zone 2'] += 1
            elif hr < 0.8 * max_hr:
                zones['Zone 3'] += 1
            elif hr < 0.9 * max_hr:
                zones['Zone 4'] += 1
            else:
                zones['Zone 5'] += 1
        return {zone: count / total_time for zone, count in zones.items()}
    else:
        return zones

# **ΠΡΟΣΟΧΗ:** Αντικαταστήστε το παρακάτω με τον εκτιμώμενο μέγιστο καρδιακό παλμό σας.
MAX_HEART_RATE = 190

query_hr_records = """
SELECT
    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,
    r.heart_rate
FROM activity a
JOIN record r ON a.activity_id = r.activity_id
WHERE a.sport = 'running' AND r.heart_rate > 0
ORDER BY a.start_time;
"""
df_hr_records = pd.read_sql_query(query_hr_records, conn)

if not df_hr_records.empty:
    hr_zone_analysis = df_hr_records.groupby('run_date')['heart_rate'].apply(list).apply(lambda x: calculate_hr_zones(x, MAX_HEART_RATE)).apply(pd.Series)
    hr_zone_analysis.index = pd.to_datetime(hr_zone_analysis.index)
    hr_zone_analysis = hr_zone_analysis.sort_index()

    hr_zone_analysis.plot(kind='bar', stacked=True, figsize=(14, 7))
    plt.title('Κατανομή Χρόνου σε Ζώνες Καρδιακών Παλμών ανά Τρέξιμο')
    plt.xlabel('Ημερομηνία')
    plt.ylabel('Ποσοστό Χρόνου')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Ζώνη Καρδιακών Παλμών')
    plt.tight_layout()
    plt.show()
else:
    print("Δεν βρέθηκαν λεπτομερή δεδομένα καρδιακών παλμών για ανάλυση ζωνών.")

# ## 3. Ανάλυση Ρυθμού και Υψομέτρου

# ### Μεταβολή Ρυθμού κατά τη Διάρκεια ενός Τρεξίματος (Επιλέξτε ένα activity_id)
# **ΠΡΟΣΟΧΗ:** Αντικαταστήστε το 'YOUR_ACTIVITY_ID' με ένα πραγματικό activity_id από τον πίνακα 'activity'.
SELECTED_ACTIVITY_ID = None # Μπορείτε να θέσετε ένα συγκεκριμένο ID για ανάλυση

if SELECTED_ACTIVITY_ID:
    query_pace_over_time = f"""
    SELECT
        strftime('%H:%M:%S', datetime(r.timestamp, 'unixepoch', 'localtime')) AS time,
        1000.0 / r.speed / 60.0 AS pace_min_km
    FROM record r
    WHERE r.activity_id = '{SELECTED_ACTIVITY_ID}' AND r.speed > 0
    ORDER BY r.timestamp;
    """
    df_pace_over_time = pd.read_sql_query(query_pace_over_time, conn)

    if not df_pace_over_time.empty:
        plt.figure(figsize=(10, 5))
        plt.plot(df_pace_over_time['time'], df_pace_over_time['pace_min_km'])
        plt.title(f'Μεταβολή Ρυθμού κατά τη Διάρκεια του Τρεξίματος (ID: {SELECTED_ACTIVITY_ID})')
        plt.xlabel('Χρόνος από την Έναρξη')
        plt.ylabel('Ρυθμός (λεπτά/km)')
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Δεν βρέθηκαν δεδομένα ρυθμού για το activity_id: {SELECTED_ACTIVITY_ID}")
else:
    print("Παρακαλώ ορίστε ένα SELECTED_ACTIVITY_ID για ανάλυση μεταβολής ρυθμού.")

# ### Συσχέτιση Ρυθμού και Υψομέτρου (Αν υπάρχουν δεδομένα υψομέτρου)

query_pace_elevation = """
SELECT
    1000.0 / r.speed / 60.0 AS pace_min_km,
    r.altitude
FROM activity a
JOIN record r ON a.activity_id = r.activity_id
WHERE a.sport = 'running' AND r.speed > 0 AND r.altitude IS NOT NULL;
"""
df_pace_elevation = pd.read_sql_query(query_pace_elevation, conn)

if not df_pace_elevation.empty:
    plt.figure(figsize=(8, 6))
    plt.scatter(df_pace_elevation['altitude'], df_pace_elevation['pace_min_km'], alpha=0.5)
    plt.title('Συσχέτιση Ρυθμού και Υψομέτρου')
    plt.xlabel('Υψόμετρο (μέτρα)')
    plt.ylabel('Ρυθμός (λεπτά/km)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Δεν βρέθηκαν δεδομένα υψομέτρου για ανάλυση συσχέτισης με τον ρυθμό.")

# ## 4. Στατιστικά Laps (Αν υπάρχουν)

query_lap_pace = """
SELECT
    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,
    l.lap_number,
    l.distance / (CAST(l.duration AS REAL) / 1000) / 60.0 AS lap_speed_kmh,
    (CAST(l.duration AS REAL) / 1000) / (l.distance / 1000.0) / 60.0 AS lap_pace_min_km
FROM activity a
JOIN lap l ON a.activity_id = l.activity_id
WHERE a.sport = 'running' AND l.distance > 0 AND l.duration > 0
ORDER BY a.start_time, l.lap_number;
"""
df_lap_pace = pd.read_sql_query(query_lap_pace, conn, parse_dates=['run_date'])

if not df_lap_pace.empty:
    plt.figure(figsize=(14, 7))
    sns.boxplot(x='lap_number', y='lap_pace_min_km', data=df_lap_pace)
    plt.title('Ρυθμός ανά Lap')
    plt.xlabel('Lap')
    plt.ylabel('Ρυθμός (λεπτά/km)')
    plt.tight_layout()
    plt.show()

    # Μέσος ρυθμός ανά lap για όλα τα τρεξίματα
    avg_lap_pace = df_lap_pace.groupby('lap_number')['lap_pace_min_km'].mean().reset_index()
    plt.figure(figsize=(10, 5))
    plt.plot(avg_lap_pace['lap_number'], avg_lap_pace['lap_pace_min_km'], marker='o', linestyle='-')
    plt.title('Μέσος Ρυθμός ανά Lap (Όλα τα Τρεξίματα)')
    plt.xlabel('Lap')
    plt.ylabel('Μέσος Ρυθμός (λεπτά/km)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Δεν βρέθηκαν δεδομένα laps για τρέξιμο.")

# ## 5. Μακροπρόθεσμες Τάσεις και Συγκρίσεις

# ### Εβδομαδιαία Συνολικά Στοιχεία (Απόσταση)

query_weekly_distance = """
SELECT
    strftime('%Y-%W', a.start_time) AS week,
    SUM(a.total_distance) / 1000.0 AS total_distance_km
FROM activity a
WHERE a.sport = 'running' AND a.total_distance > 0
GROUP BY week
ORDER BY week;
"""
df